# SVGP two-stage double-check (local)

This notebook **double-checks** a *single replication*'s data-dependent $\alpha^*(x)$ against the true noiseless latent function $f_0(x)$.

## What was updated to match the **new** pipeline

1. **Input standardization (A1 / mentor-style)**
   - $x_{01} = (x - \text{train\_low})/(\text{train\_high}-\text{train\_low}) \in [0,1]$
   - $x_{std} = 2x_{01}-1 \in [-1,1]$
   - Applied consistently to **train/test/Z**.

2. **Target standardization**
   - Training uses **Z-score** targets: $y_{std}=(y-\bar y)/s_y$ with $(\bar y, s_y)$ computed on the **training pool**.
   - Predictions are **de-standardized** back to raw $y$ for MSE, coverage, and diagnostics:
     - $\mu = \mu_{std}\,s_y + \bar y$
     - $\mathrm{Var}(f)=\mathrm{Var}_{std}(f)\,s_y^2$

3. **Nugget/noise conversion**
   - You still use a fixed GaussianLikelihood noise as a *nugget* for VI stability.
   - The configured `noise_raw` is interpreted in **raw y** units, and converted to standardized units via:
     - `noise_std = noise_raw / (y_std**2)`

4. **Training inputs generation alignment**
   - The training **pool** is drawn **once** (stratified/LHS-like) with base seed `cfg.seed`.
   - For double-check you train on the **full pool** (ordering irrelevant). No re-draw per replication.

5. **CI level handling**
   - Uses `ci_level` to compute $z_{ci}=\Phi^{-1}((1+\text{ci\_level})/2)$ dynamically (no hard-coded 1.96).

## Outputs
This notebook produces **one CSV per replication** (here, `rep_id=54` by default), and does **not** average across reps.


In [11]:
import os, math, time
from dataclasses import dataclass
from contextlib import ExitStack

import torch
import gpytorch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

def now():
    return time.strftime('%H:%M:%S')

def log(msg: str):
    print(f"[{now()}] {msg}", flush=True)

def has_setting(name: str) -> bool:
    return hasattr(gpytorch.settings, name)

def ciq_context():
    stack = ExitStack()
    for name, val in [
        ("num_contour_quadrature", 15),
        ("ciq_samples", 16),
        ("ciq_max_iter", 30),
        ("ciq_forward_precision", 0.01),
        ("eval_cg_tolerance", 1e-4),
        ("max_root_decomposition_size", 64),
    ]:
        if has_setting(name):
            stack.enter_context(getattr(gpytorch.settings, name)(val))
    stack.enter_context(gpytorch.settings.fast_pred_var(True))
    return stack

def clone_state_dict(sd):
    out = {}
    for k, v in sd.items():
        out[k] = v.detach().clone() if torch.is_tensor(v) else v
    return out

def f0(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * math.pi * x) + 0.5 * torch.cos(3 * x)

def make_stratified_points_1d(n, low, high, generator, device, dtype):
    i = torch.arange(n, device=device, dtype=dtype)
    u = (i + torch.rand(n, generator=generator, device=device, dtype=dtype)) / n
    x = low + (high - low) * u
    perm = torch.randperm(n, generator=generator, device=device)
    return x[perm].unsqueeze(-1)


In [12]:
@dataclass
class CFG:
    # data
    n_train: int = 500
    n_test: int = 50
    train_low: float = -2.5
    train_high: float = 2.5
    test_low: float = -2.5
    test_high: float = 2.5

    # model / training (keep consistent with your alpha-suite)
    use_fp64: bool = True
    use_ciq: bool = False
    inducing_points: int = 50
    stage1_epochs: int = 2000
    stage2_epochs: int = 800
    batch_size: int = 500
    lr_adam_stage1: float = 2e-2
    lr_adam_stage2: float = 2e-2
    jitter: float = 1e-2
    seed: int = 13

    # nugget in RAW y units (converted to standardized units inside training)
    noise_raw: float = 3e-3

    # CI
    ci_level: float = 0.95

cfg = CFG()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float64 if cfg.use_fp64 else torch.float32)
log(f"Device: {device} | dtype: {torch.get_default_dtype()}")


[23:21:22] Device: cuda | dtype: torch.float64


In [13]:
# ---------------- SVGP model & training helpers ----------------

class SVGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points_std: torch.Tensor):
        M = inducing_points_std.size(0)
        q_dist = gpytorch.variational.CholeskyVariationalDistribution(M)
        q_strat = gpytorch.variational.VariationalStrategy(
            self, inducing_points_std, q_dist, learn_inducing_locations=False
        )
        super().__init__(q_strat)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x_std):
        mean_x = self.mean_module(x_std)
        covar_x = self.covar_module(x_std)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

def freeze_all_params(model, likelihood):
    for p in model.parameters():
        p.requires_grad_(False)
    for p in likelihood.parameters():
        p.requires_grad_(False)

def pick_covariance_params_svgp(model):
    """Select ONLY variational covariance parameters (exclude any '*mean*')."""
    trainables, names = [], []
    vdist_mod = None
    vs = getattr(model, 'variational_strategy', None)
    if vs is not None:
        if hasattr(vs, '_variational_distribution'):
            vdist_mod = getattr(vs, '_variational_distribution')
        else:
            cand = getattr(vs, 'variational_distribution', None)
            if isinstance(cand, torch.nn.Module):
                vdist_mod = cand
    if isinstance(vdist_mod, torch.nn.Module):
        for n, p in vdist_mod.named_parameters(recurse=True):
            low = n.lower()
            if 'mean' in low:
                p.requires_grad_(False)
            elif any(k in low for k in ['chol', 'std', 'var', 'covar', 'factor']):
                p.requires_grad_(True)
                trainables.append(p)
                names.append(f"vdist.{n}")
            else:
                p.requires_grad_(False)
    if not trainables:
        allow = (
            'chol_variational_covar','variational_distribution.chol_covar',
            'variational_distribution._cholesky_factor','_variational_distribution.chol_covar',
            'raw_covar','chol_factor','covar_factor','variational_stddev','raw_var',
            'qchol','q_scale_tril'
        )
        for n, p in model.named_parameters():
            low = n.lower()
            if 'variational_strategy' in low and any(k in low for k in allow) and 'mean' not in low:
                p.requires_grad_(True)
                trainables.append(p)
                names.append(n)
    return trainables, names

def stage1_train(model, likelihood, x_train_std, y_train_std, lr, epochs, jitter,
                 batch_size):
    N = x_train_std.size(0)
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=N, beta=1.0)

    likelihood.noise_covar.raw_noise.requires_grad_(False)
    opt = torch.optim.Adam([
        {'params': model.variational_parameters()},
        {'params': model.hyperparameters()},
        {'params': likelihood.parameters()},
    ], lr=lr)

    loader = DataLoader(TensorDataset(x_train_std, y_train_std),
                        batch_size=batch_size, shuffle=True, drop_last=False)

    model.train(); likelihood.train()
    best_loss, best_epoch = float('inf'), 0
    best_model_sd = clone_state_dict(model.state_dict())
    best_lik_sd   = clone_state_dict(likelihood.state_dict())

    hist = []
    with gpytorch.settings.cholesky_jitter(jitter):
        for ep in range(1, epochs + 1):
            total, n_batches = 0.0, 0
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                out = model(xb)
                loss = -mll(out, yb)
                if loss.numel() > 1:
                    loss = loss.mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                total += float(loss.item()); n_batches += 1
            avg_loss = total / max(1, n_batches)
            hist.append(avg_loss)
            if avg_loss < best_loss:
                best_loss, best_epoch = avg_loss, ep
                best_model_sd = clone_state_dict(model.state_dict())
                best_lik_sd   = clone_state_dict(likelihood.state_dict())

    model.load_state_dict(best_model_sd)
    likelihood.load_state_dict(best_lik_sd)
    model.eval(); likelihood.eval()
    return hist, best_epoch, best_loss

def stage2_train_cov_only(model, likelihood, x_train_std, y_train_std, beta, lr,
                          epochs, jitter, batch_size):
    N = x_train_std.size(0)
    freeze_all_params(model, likelihood)
    cov_params, cov_names = pick_covariance_params_svgp(model)
    assert cov_params, 'No covariance params found to train in Stage-2.'

    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=N, beta=beta)
    opt = torch.optim.Adam(cov_params, lr=lr)

    loader = DataLoader(TensorDataset(x_train_std, y_train_std),
                        batch_size=batch_size, shuffle=True, drop_last=False)

    model.train(); likelihood.train()
    best_loss, best_epoch = float('inf'), 0
    best_model_sd = clone_state_dict(model.state_dict())
    best_lik_sd   = clone_state_dict(likelihood.state_dict())

    hist = []
    with gpytorch.settings.cholesky_jitter(jitter):
        for ep in range(1, epochs + 1):
            total, n_batches = 0.0, 0
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                out = model(xb)
                loss = -mll(out, yb)
                if loss.numel() > 1:
                    loss = loss.mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(cov_params, 5.0)
                opt.step()
                total += float(loss.item()); n_batches += 1
            avg_loss = total / max(1, n_batches)
            hist.append(avg_loss)
            if avg_loss < best_loss:
                best_loss, best_epoch = avg_loss, ep
                best_model_sd = clone_state_dict(model.state_dict())
                best_lik_sd   = clone_state_dict(likelihood.state_dict())

    model.load_state_dict(best_model_sd)
    likelihood.load_state_dict(best_lik_sd)
    model.eval(); likelihood.eval()
    return hist, best_epoch, best_loss, cov_names


In [14]:
# ---------------- User inputs (paths) ----------------

# Calibration CSV (Windows path). If you run this notebook locally, keep this as-is.
calib_csv_path = r"C:\\Users\\yangy\\26 spring\\double check\\pointwise_coverage_rep054_A7_B500.csv"

# Output directory for double-check results
out_dir = r"C:\\Users\\yangy\\26 spring\\double check\\doublecheck_outputs"
os.makedirs(out_dir, exist_ok=True)

# Which replication this CSV corresponds to
rep_id = 54

log(f"Calibration CSV: {calib_csv_path}")
log(f"Output dir:      {out_dir}")
log(f"rep_id:         {rep_id}")


[23:21:22] Calibration CSV: C:\\Users\\yangy\\26 spring\\double check\\pointwise_coverage_rep054_A7_B500.csv
[23:21:22] Output dir:      C:\\Users\\yangy\\26 spring\\double check\\doublecheck_outputs
[23:21:22] rep_id:         54


In [15]:
# ---------------- Load calibration dictionary & compute alpha_star(x) ----------------

df = pd.read_csv(calib_csv_path)
assert df.columns[0].lower().startswith('x'), "First column must be x"
x_axis = df.iloc[:, 0].to_numpy()  # [J]
J = x_axis.shape[0]
log(f"Loaded calibration CSV with J={J} test points")

alpha_cols = [c for c in df.columns if c.startswith('alpha_')]
assert len(alpha_cols) > 0, "No alpha_* columns found"

alpha_values = np.array([float(c.split('_')[1]) for c in alpha_cols], dtype=float)  # [A]
coverage_mat = df[alpha_cols].to_numpy()  # [J, A]
A = alpha_values.size
log(f"Found A={A} alphas: {alpha_values}")

target = cfg.ci_level  # pick alpha to reach coverage ~ ci_level
alpha_star = np.zeros(J, dtype=float)
cov_at_alpha_star_calib = np.zeros(J, dtype=float)

order_desc = np.argsort(alpha_values)[::-1]
alphas_desc = alpha_values[order_desc]

for j in range(J):
    cov_row = coverage_mat[j, :]
    cov_desc = cov_row[order_desc]
    idx_ge = np.where(cov_desc >= target)[0]
    if idx_ge.size > 0:
        chosen_desc_idx = idx_ge[0]   # largest alpha that still reaches target
    else:
        chosen_desc_idx = int(np.argmax(cov_desc))  # max coverage if none reaches target
    best_idx = int(order_desc[chosen_desc_idx])
    alpha_star[j] = alpha_values[best_idx]
    cov_at_alpha_star_calib[j] = cov_row[best_idx]

log('Example alpha_star (first 8 test points):')
for j in range(min(8, J)):
    log(f"  j={j:02d}, x={x_axis[j]: .6f}, alpha*={alpha_star[j]:.4f}, cov@alpha*={cov_at_alpha_star_calib[j]:.3f}")

unique_alphas = np.unique(alpha_star)
log(f"Unique alpha_star values: {unique_alphas}")


[23:21:22] Loaded calibration CSV with J=50 test points
[23:21:22] Found A=7 alphas: [1.    0.8   0.6   0.5   0.3   0.1   0.001]
[23:21:22] Example alpha_star (first 8 test points):
[23:21:22]   j=00, x=-2.500000, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=01, x=-2.397959, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=02, x=-2.295918, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=03, x=-2.193878, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=04, x=-2.091837, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=05, x=-1.989796, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=06, x=-1.887755, alpha*=1.0000, cov@alpha*=1.000
[23:21:22]   j=07, x=-1.785714, alpha*=1.0000, cov@alpha*=1.000
[23:21:22] Unique alpha_star values: [1.]


In [16]:
# ---------------- Rebuild train/test + apply NEW standardization ----------------

# Base seed used to generate the *shared training pool* (aligned with your new pipeline)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

gen = torch.Generator(device=device).manual_seed(cfg.seed)

dtype = torch.get_default_dtype()
x_train_raw = make_stratified_points_1d(cfg.n_train, cfg.train_low, cfg.train_high, gen, device, dtype)
y_train_raw = f0(x_train_raw).squeeze(-1)

# Test grid MUST match CSV x values
x_test_raw = torch.tensor(x_axis, device=device, dtype=dtype).unsqueeze(1)
y_test_true_raw = f0(x_test_raw).squeeze(-1)

# A1 x normalization: minmax -> [0,1] then -> [-1,1]
train_low = torch.as_tensor(cfg.train_low, device=device, dtype=dtype)
train_high = torch.as_tensor(cfg.train_high, device=device, dtype=dtype)
span = (train_high - train_low).clamp_min(1e-12)

def x_to_std(x_raw: torch.Tensor) -> torch.Tensor:
    x01 = (x_raw - train_low) / span
    return 2.0 * x01 - 1.0

x_train_std = x_to_std(x_train_raw)
x_test_std = x_to_std(x_test_raw)

# Fixed inducing points (raw evenly spaced), transformed with same x_to_std
Z_raw = torch.linspace(cfg.train_low, cfg.train_high, cfg.inducing_points, device=device, dtype=dtype).unsqueeze(1)
Z_std = x_to_std(Z_raw)

left_std = float(x_to_std(torch.tensor([[cfg.train_low]], device=device, dtype=dtype)))
right_std = float(x_to_std(torch.tensor([[cfg.train_high]], device=device, dtype=dtype)))
log(f"MinMax norm: raw [{cfg.train_low:.2f},{cfg.train_high:.2f}] -> std [{left_std:.2f},{right_std:.2f}] (≈ -1,+1)")

# y standardization (Z-score) computed on training pool
y_mean = y_train_raw.mean()
y_std = y_train_raw.std(unbiased=False).clamp_min(1e-12)
y_train_std = (y_train_raw - y_mean) / y_std

log(f"y_mean={float(y_mean):.6f}, y_std={float(y_std):.6f}")


[23:21:22] MinMax norm: raw [-2.50,2.50] -> std [-1.00,1.00] (≈ -1,+1)
[23:21:22] y_mean=0.062575, y_std=0.792548


In [17]:
# ---------------- Compute z_ci from ci_level ----------------

norm = torch.distributions.Normal(
    torch.tensor(0.0, device=device, dtype=dtype),
    torch.tensor(1.0, device=device, dtype=dtype)
)
z_ci = float(norm.icdf(torch.tensor([(1.0 + cfg.ci_level) / 2.0], device=device, dtype=dtype))[0].item())
log(f"Using z_ci={z_ci:.6f} for ci_level={cfg.ci_level:.3f}")


[23:21:22] Using z_ci=1.959964 for ci_level=0.950


In [18]:
# ---------------- Stage-1 on full pool (alpha=1) ----------------

model = SVGPModel(Z_std.clone().contiguous()).to(device=device)
likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device=device)

with torch.no_grad():
    model.covar_module.base_kernel.lengthscale.fill_(1.0)
    model.covar_module.outputscale.fill_(1.0)
    # convert nugget from raw-y units to standardized-y units
    likelihood.noise = torch.as_tensor(cfg.noise_raw, device=device, dtype=dtype) / (y_std ** 2)

likelihood.noise_covar.raw_noise.requires_grad_(False)

with (ciq_context() if cfg.use_ciq else ExitStack()):
    _, best_epoch_s1, best_loss_s1 = stage1_train(
        model, likelihood, x_train_std, y_train_std,
        lr=cfg.lr_adam_stage1, epochs=cfg.stage1_epochs,
        jitter=cfg.jitter, batch_size=cfg.batch_size
    )

with torch.no_grad():
    pred_std = model(x_test_std)
    mu_std = pred_std.mean
    mu_raw = mu_std * y_std + y_mean
    mse_s1 = torch.mean((mu_raw - y_test_true_raw) ** 2).item()

log(f"[rep {rep_id:03d}] Stage-1 (α=1.000) | best -ELBO={best_loss_s1:.6f} @epoch {best_epoch_s1} | MSE={mse_s1:.6f}")

s1_model_sd = clone_state_dict(model.state_dict())
s1_lik_sd   = clone_state_dict(likelihood.state_dict())


[23:21:52] [rep 054] Stage-1 (α=1.000) | best -ELBO=-1.518500 @epoch 1998 | MSE=0.000040


In [19]:
# ---------------- Stage-2 for each unique alpha_star, compute indicators ----------------

alpha_to_idxs = {a: np.where(alpha_star == a)[0] for a in unique_alphas}
indicators = np.zeros(J, dtype=np.float64)

for alpha in unique_alphas:
    idxs_np = alpha_to_idxs[alpha]
    idxs_torch = torch.from_numpy(idxs_np).to(device=device, dtype=torch.long)
    beta = float(1.0 / float(alpha))

    model.load_state_dict(s1_model_sd)
    likelihood.load_state_dict(s1_lik_sd)
    model.eval(); likelihood.eval()

    with (ciq_context() if cfg.use_ciq else ExitStack()):
        _, best_epoch_s2, best_loss_s2, _ = stage2_train_cov_only(
            model, likelihood, x_train_std, y_train_std,
            beta=beta, lr=cfg.lr_adam_stage2, epochs=cfg.stage2_epochs,
            jitter=cfg.jitter, batch_size=cfg.batch_size
        )

    with torch.no_grad():
        pred_std = model(x_test_std)
        mu_std = pred_std.mean
        var_std = pred_std.variance.clamp_min(1e-24)

        # de-standardize to raw y-space
        mu_raw = mu_std * y_std + y_mean
        var_raw = var_std * (y_std ** 2)
        std_raw = var_raw.sqrt().clamp_min(1e-12)

        lower = mu_raw[idxs_torch] - z_ci * std_raw[idxs_torch]
        upper = mu_raw[idxs_torch] + z_ci * std_raw[idxs_torch]
        y_sel = y_test_true_raw[idxs_torch]

        cover = ((y_sel >= lower) & (y_sel <= upper)).to(torch.float64)
        indicators[idxs_np] = cover.cpu().numpy()

        mse = torch.mean((mu_raw - y_test_true_raw) ** 2).item()

    log(f"[rep {rep_id:03d}] α={float(alpha):.4f} (Stage-2) | best -ELBO={best_loss_s2:.6f} @epoch {best_epoch_s2} | MSE={mse:.6f} | num_points={len(idxs_np)}")


[23:22:04] [rep 054] α=1.0000 (Stage-2) | best -ELBO=-1.521411 @epoch 112 | MSE=0.000040 | num_points=50


In [20]:
# ---------------- Save per-rep CSV (NO averaging across reps) ----------------

out_csv = os.path.join(out_dir, f"rep_{rep_id:03d}_doublecheck.csv")
df_out = pd.DataFrame({
    'x': x_axis,
    'alpha_star': alpha_star,
    'cov_indicator': indicators,
    'cov_at_alpha_star_calib': cov_at_alpha_star_calib,
})
df_out.to_csv(out_csv, index=False)

log(f"Saved: {out_csv}")
log("Head:")
print(df_out.head(10).to_string(index=False))

# Optional: quick sanity summary
log(f"Mean double-check coverage (this rep): {df_out['cov_indicator'].mean():.4f}")


[23:22:04] Saved: C:\\Users\\yangy\\26 spring\\double check\\doublecheck_outputs\rep_054_doublecheck.csv
[23:22:04] Head:
        x  alpha_star  cov_indicator  cov_at_alpha_star_calib
-2.500000         1.0            1.0                      1.0
-2.397959         1.0            1.0                      1.0
-2.295918         1.0            1.0                      1.0
-2.193878         1.0            1.0                      1.0
-2.091837         1.0            1.0                      1.0
-1.989796         1.0            1.0                      1.0
-1.887755         1.0            1.0                      1.0
-1.785714         1.0            1.0                      1.0
-1.683673         1.0            1.0                      1.0
-1.581633         1.0            1.0                      1.0
[23:22:04] Mean double-check coverage (this rep): 1.0000
